In [ ]:
# Notebook: Power Spectrum of a Triangular Signal
# Author: Adapted for Springer-style presentation

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, BoundedIntText, FloatSlider

# ==========================
# Parameters
# ==========================

t = np.linspace(-np.pi, np.pi, 2000)


# ==========================
# Triangular signal
# ==========================

def x(t, E=1):
    return np.where(
        t < 0,
        E + E*t/np.pi,
        E - E*t/np.pi
    )


# ==========================
# Fourier reconstruction (time domain)
# ==========================

def x_fourier(t, N, E=1):
    xf = E/2

    for k in range(N):
        n = 2*k + 1
        xf += (4*E/np.pi**2)*np.cos(n*t)/n**2

    return xf


# ==========================
# Interactive Plot Function
# ==========================

@interact(
    N=BoundedIntText(value=5, min=1, max=20, description="N"),
    E=FloatSlider(value=1, min=0.1, max=5, step=0.1, description="E")
)
def plot_power_spectrum(N, E):
    # Time domain calculations
    xx = x(t, E)
    xf = x_fourier(t, N, E)

    # Frequency domain calculations (Power Spectrum)
    n_values = np.arange(1, 2 * N + 1, 2)
    
    # Υπολογισμός ισχύος για κάθε αρμονική (τετράγωνο του πλάτους C_n^2)
    # DC όρος: ισχύς = (E/2)^2
    # Αρμονικές: ισχύς = C_n^2 = (4*E / (pi^2 * n^2))^2
    dc_power = (E / 2) ** 2
    powers = np.zeros_like(n_values, dtype=float)
    for idx, n in enumerate(n_values):
        cn = (4 * E) / (np.pi**2 * n**2)
        powers[idx] = cn ** 2

    # Δημιουργία υπογραφήματος 2 γραμμών
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6.5))

    # --- 1. Time Domain Plot ---
    ax1.plot(t, xx, 'r', linewidth=2.5, label="Original signal")
    ax1.plot(t, xf, 'b', linewidth=2, label=f"Fourier expansion (N={N})")
    ax1.set_xlabel(r"$\omega t$")
    ax1.set_ylabel("Amplitude")
    ax1.grid(True)
    ax1.legend(loc='upper right')
    ax1.set_title("Triangular Signal Fourier Reconstruction (Time Domain)")

    # --- 2. Power Spectrum Plot (Stem plot με τετραγωνισμένες τιμές) ---
    spectrum_n = np.concatenate(([0], n_values))
    spectrum_P = np.concatenate(([dc_power], powers))
    
    markerline, stemlines, baseline = ax2.stem(
        spectrum_n, spectrum_P, 
        basefmt="k-", linefmt='g-', markerfmt='go' # Πράσινο χρώμα για διάκριση από το amplitude
    )
    
    # Εκτύπωση της τιμής ισχύος πάνω από κάθε γραμμή
    for n_val, p_val in zip(spectrum_n, spectrum_P):
        ax2.text(
            n_val, p_val + 0.02 * max(spectrum_P), 
            f"{p_val:.3f}", 
            ha='center', 
            va='bottom', 
            fontsize=9,
            fontweight='bold',
            color='darkgreen'
        )

    ax2.set_xlabel("Harmonic Number ($n$)")
    ax2.set_ylabel("Power ($C_n^2$)")
    ax2.grid(True)
    ax2.set_title("Discrete Power Spectrum")
    
    # Προσαρμογή ορίων άξονα y
    ax2.set_ylim(0, max(spectrum_P) * 1.15)
    ax2.set_xticks(spectrum_n)

    plt.tight_layout()
    plt.show()
    plt.close(fig)